## As a data scientist working for the front office of a major American multinational bank, you are responsible for enhancing customer service and ensuring compliance with financial regulations. Your current assignment involves analysing the customer complaints the bank has received over the past year.

## The current time-consuming manual process for daily triaging and reviewing of customer complaintsThe complaints data is currently underutilized in enhancing the quality of products and services.

## The goal is to use NLP techniques, such as text classification and sentiment analysis, to efficiently gain insights into the underlying causes of customer grievances. By leveraging these methods, we aim to better understand and address customer grievances, ultimately improving our grievance redressal process.

1. Read data in python environment.
2. Check if the variables have correct datatypes. Make changes wherever necessary.
3. Find the date range
4. Define a function named preprocessing that executes the following series of pre-processing steps in order:
- Convert text to lowercase
- Remove numbers
- Remove stopwords
- Remove punctuation
- Apply lemmatization
5. Clean the text under ‘Complaint Description’ using the above function
6. Convert the pre-processed text into a matrix of TF-IDF features for downstream modelling.
7. In order to effectively manage the process, it is critically important to categorise the complaint and pass on to the concerned product department. Consider the department as a target variable and build a classification model.
8. Use any transformer-based model to do the same task(as 7)
9. Use SentimentIntensityAnalyzer  to predict sentiments from the complaints. The SentimentIntensityAnalyzer is a class from the vaderSentiment library designed for sentiment analysis. It evaluates text to determine the sentiment scores across four categories: positive, negative, neutral, and compound. The compound score is a normalized value between -1 (most extreme negative) and +1 (most extreme positive), providing an overall sentiment rating. This analyzer is particularly effective for social media and other informal texts, as it can interpret emoticons, acronyms, and slang. It is widely used for tasks like sentiment classification, opinion mining, and customer feedback analysis. Its ease of use and accuracy make it a valuable tool in NLP.
10. How can the score be used by the bank? Share your insights.

In [30]:
# 1. Read data in python environment.
# 2. Check if the variables have correct datatypes. Make changes wherever necessary.
# 3. Find the date range

import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import os
os.environ["USE_TF"] = "0" 

# 1. Read data in python environment.
# Load the CSV file
df = pd.read_csv("banking_complaints.csv")

#Preview data
print(df.head())

# 2. Check if the variables have correct datatypes. Make changes wherever necessary.
print(df.dtypes)

#Convert 'Date Received' column to datetime format
df['Date Received'] = pd.to_datetime(df['Date Received'], errors='coerce')

#data types to confirm
print(df.dtypes)

# Find the date range
#Find the earliest and latest complaint date
min_date = df['Date Received'].min()
max_date = df['Date Received'].max()

print("Earliest complaint date:", min_date)
print("Latest complaint date:", max_date)

# 4. Define a function named preprocessing that executes the following series of pre-processing steps in order:
# - Convert text to lowercase
# - Remove numbers
# - Remove stopwords
# - Remove punctuation
# - Apply lemmatization

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')  


# Stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocessing(text):
    #Convert to lowercase
    text = text.lower()
    
    #Remove numbers
    text = re.sub(r'\d+', '', text)
    
    #Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    #Tokenize and remove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    
    #Lemmatize
    lemmatized = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Join back into a single string
    return ' '.join(lemmatized)

# 5. Clean the text under ‘Complaint Description’ using the above function
df['Cleaned Complaint Description'] = df['Complaint Description'].astype(str).apply(preprocessing)

#results
print(df[['Complaint Description', 'Cleaned Complaint Description']].head())

# 6. Convert the pre-processed text into a matrix of TF-IDF features for downstream modelling.
from sklearn.feature_extraction.text import TfidfVectorizer

#Initialize the TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)  

#Fit and transform the cleaned text
X_tfidf = tfidf_vectorizer.fit_transform(df['Cleaned Complaint Description'])

#DataFrame for inspection 
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

# Preview
print(tfidf_df.head())


# 7. In order to effectively manage the process, it is critically important to categorise the complaint and pass on to the concerned product department. Consider the department as a target variable and build a classification model.

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

#Define features and target
X = X_tfidf
y = df['Banking Product']

#Split data into training and testing sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Train a classifier (Logistic Regression for baseline)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

#Predict on the test set
y_pred = model.predict(X_test)

#Evaluate model performance
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

  Complaint ID Date Received  \
0  CID76118977      1/1/2023   
1  CID98703933      1/1/2023   
2  CID52036665      1/1/2023   
3  CID62581335      1/1/2023   
4  CID65731164      1/1/2023   

                                     Banking Product   Issue ID  \
0                        Checking or savings account  I_3510635   
1  Credit reporting, credit repair services, or o...  I_3798538   
2                        Checking or savings account  I_3648593   
3                        Credit card or prepaid card  I_6999080   
4                        Checking or savings account  I_3648593   

                               Complaint Description       State    ZIP  \
0  on XX/XX/XX22 I opened a safe balance account ...  California  92311   
1  There is an item from Bank of ABC on my credit...  California  91344   
2  On XX/XX/XX22 I found out that my account was ...    New York  10466   
3  I've had a credit card for years with Bank of ...  California  92127   
4  This issue has to do with 

[nltk_data] Downloading package stopwords to /Users/home/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/home/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/home/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


                               Complaint Description  \
0  on XX/XX/XX22 I opened a safe balance account ...   
1  There is an item from Bank of ABC on my credit...   
2  On XX/XX/XX22 I found out that my account was ...   
3  I've had a credit card for years with Bank of ...   
4  This issue has to do with the way that Bank of...   

                       Cleaned Complaint Description  
0  xxxxxx opened safe balance account online usin...  
1  item bank abc credit report belong must remove...  
2  xxxxxx found account frozen apparent reason we...  
3  ive credit card year bank abc xxxxxxxx paid ba...  
4  issue way bank abc account linking bill pay pa...  
   aaa  aba  abandoned       abc  abcmerill  abcn  abcns  abcxxxx  abide  \
0  0.0  0.0        0.0  0.000000        0.0   0.0    0.0      0.0    0.0   
1  0.0  0.0        0.0  0.069283        0.0   0.0    0.0      0.0    0.0   
2  0.0  0.0        0.0  0.000000        0.0   0.0    0.0      0.0    0.0   
3  0.0  0.0        0.0  0.022

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  

In [ ]:
# 8. Use any transformer-based model to do the same task(as 7)
import os
os.environ["USE_TF"] = "0" 

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch
from datasets import Dataset
import torch

device = torch.device("cpu")
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
).to(device)

#Encode labels (target variable)
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['Banking Product'])

#Train-test split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['Complaint Description'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

#okenization
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
train_encodings = tokenizer(train_texts, truncation=True, padding=True)
test_encodings = tokenizer(test_texts, truncation=True, padding=True)

#PyTorch Dataset wrapper
class ComplaintDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        return {
            key: torch.tensor(val[idx]) for key, val in self.encodings.items()
        } | {'labels': torch.tensor(self.labels[idx])}

    def __len__(self):
        return len(self.labels)

train_dataset = ComplaintDataset(train_encodings, train_labels)
test_dataset = ComplaintDataset(test_encodings, test_labels)

#Load model
num_labels = len(label_encoder.classes_)
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=num_labels)

#Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    logging_dir='./logs',
    logging_steps=10
)

#Metrics function
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    labels = pred.label_ids
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted')
    }

#Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

#Train the model
trainer.train()

# Evaluate on test set
results = trainer.evaluate()
print("Test Set Results:", results)



In [31]:
#Use SentimentIntensityAnalyzer  to predict sentiments from the complaints. The SentimentIntensityAnalyzer is a class from the vaderSentiment library designed for sentiment analysis. It evaluates text to determine the sentiment scores across four categories: positive, negative, neutral, and compound. The compound score is a normalized value between -1 (most extreme negative) and +1 (most extreme positive), providing an overall sentiment rating. This analyzer is particularly effective for social media and other informal texts, as it can interpret emoticons, acronyms, and slang. It is widely used for tasks like sentiment classification, opinion mining, and customer feedback analysis. Its ease of use and accuracy make it a valuable tool in NLP.

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initialize the sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

def get_sentiment_scores(text):
    scores = analyzer.polarity_scores(text)
    return scores

# Apply to complaint descriptions
sentiment_scores = df['Complaint Description'].astype(str).apply(get_sentiment_scores)

# Convert list of dicts to DataFrame
sentiment_df = pd.DataFrame(sentiment_scores.tolist())

# Merge back with original DataFrame
df = pd.concat([df, sentiment_df], axis=1)

def categorize_sentiment(compound_score):
    if compound_score >= 0.05:
        return 'positive'
    elif compound_score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['Sentiment Label'] = df['compound'].apply(categorize_sentiment)


# 10.

The sentiment scores from customer complaints help the bank improve service, fix product issues, and stay compliant with regulations. When a complaint has a very negative score, the bank can treat it as urgent and respond faster. By looking at sentiment across different products, the bank can see which areas are causing the most frustration. Over time, this helps the bank find patterns and fix problems before they get worse. The scores also help teams understand how customers feel after a complaint is handled. If many people feel positive, it shows what is working. If many feel negative, it shows where to improve. These scores can also be used in models to predict customer churn or risk. Overall, sentiment scores help the bank act faster, make better decisions, and keep customers happier.